# Decision Trees, Explained Simply
A flowchart of yes/no questions, learned from data. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A bank has 500 past loan applications: income (thousands of dollars), debt ratio (debt divided by income), years employed, whether they defaulted before, and whether the loan was approved.

### 500 loan applications
- `loans.csv`: 500 applications: income_k, debt_ratio, years_employed, previous_default (1 = yes), approved (1 = yes)

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "decision-trees/loans.csv")
df = pd.read_csv(url)
print(df.shape)
print(df["approved"].value_counts().to_dict())
df.head()

## 2. Gini impurity
**Gini impurity = 1 - (sum of each class share squared)**. 0 means pure (all one class); 0.5 means a 50/50 mix of two classes.

### Gini by hand
- `c / total`: the share of each class in the group
- `1 - sum(share squared)`: the Gini impurity
- `after the split`: the weighted average: 4 of the 10 go left, 6 go right

In [ ]:
def gini(counts):
    total = sum(counts)
    return 1 - sum((c / total) ** 2 for c in counts)

print("5 approved, 5 rejected:", round(gini([5, 5]), 3))
print("4 approved, 0 rejected:", round(gini([4, 0]), 3))
print("1 approved, 5 rejected:", round(gini([1, 5]), 3))
print("after the split:       ", round(0.4 * gini([4, 0]) + 0.6 * gini([1, 5]), 3))

## 3. Train a tree
scikit-learn tries every input and every threshold, and picks the question that lowers the impurity most. Then it repeats inside each branch.

### A tree with depth 3
- `max_depth=3`: at most 3 questions in a row
- `no scaling`: trees compare one input with a threshold, so units don't matter

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = df.drop(columns="approved")
y = df["approved"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
print("train accuracy:", round(tree.score(X_train, y_train), 3))
print("test accuracy: ", round(tree.score(X_test, y_test), 3))

### Draw the tree
- `plot_tree`: draws the flowchart
- `each box`: the question, how many training applicants reached it, and the class split
- `colour`: orange leans rejected, blue leans approved; darker means purer

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=X.columns, class_names=["rejected", "approved"],
          filled=True, rounded=True, fontsize=12, impurity=False)
plt.show()

### The tree as rules
- `export_text`: the same tree, written as indented if-then rules
- `[:12]`: just the first 12 lines: the left half of the tree
- `class: 1`: approved; class: 0 means rejected

In [ ]:
from sklearn.tree import export_text

rules = export_text(tree, feature_names=list(X.columns), decimals=2)
print("\n".join(rules.split("\n")[:12]))

## 4. Depth and overfitting
A deeper tree can always fit the training data better, until it memorises it.

### Train and test accuracy by depth
- `None`: no limit: grow until every leaf is pure
- `leaves`: the number of final answer boxes
- `the output`: training accuracy climbs to 100%, test accuracy peaks at depth 3, then falls: overfitting

In [ ]:
for d in [1, 2, 3, 4, 6, 10, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    print(f"depth {str(d):>4}: train {t.score(X_train, y_train):.3f}  "
          f"test {t.score(X_test, y_test):.3f}  leaves {t.get_n_leaves()}")

## 5. Which inputs matter?
**Feature importance**: how much each input lowered the impurity, across all the questions that used it.

### Feature importance
- `feature_importances_`: one number per input; they add up to 1
- `the chart`: the inputs the tree's questions rely on most

In [ ]:
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values()
imp.plot.barh()
plt.xlabel("importance"); plt.title("What the tree relies on")
plt.show()
print(imp.round(2).to_dict())

## 6. Mini project: explain a decision
Follow one applicant down the tree.

### Why was this applicant approved?
- `decision_path`: the boxes this applicant passes through
- `t.feature[n], t.threshold[n]`: the question asked at each box
- `the output`: every question and answer on the way to the decision: a full explanation

In [ ]:
applicant = pd.DataFrame([{"income_k": 85, "debt_ratio": 0.45,
                           "years_employed": 6, "previous_default": 0}])
node_ids = tree.decision_path(applicant).indices
t = tree.tree_
for n in node_ids[:-1]:
    name, thr = X.columns[t.feature[n]], t.threshold[n]
    value = applicant.iloc[0][name]
    print(f"{name} = {value}  {'<=' if value <= thr else '>'} {thr:.2f}")
print("decision:", ["rejected", "approved"][tree.predict(applicant)[0]])

## Try it yourself
1. Follow the tree by hand for an applicant with income 50k, debt ratio 0.2, 10 years employed, no previous default.
2. Try min_samples_leaf=20 with no max_depth. How do train and test accuracy compare?
3. Work out the Gini impurity of a group with 3 approved and 7 rejected applicants.